# Advanced EDA: E-Commerce Customer Analytics

A comprehensive exploratory data analysis of a realistic e-commerce dataset with advanced techniques, statistical testing, and actionable insights.

## Part 1: Data Generation & Exploration

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Generate realistic e-commerce dataset
np.random.seed(42)

n_records = 5000
date_range = pd.date_range('2023-01-01', periods=n_records, freq='8H')

df = pd.DataFrame({
    'transaction_id': range(1, n_records + 1),
    'customer_id': np.random.randint(1000, 5000, n_records),
    'timestamp': date_range,
    'product_category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Beauty', 'Sports'], n_records, p=[0.25, 0.3, 0.2, 0.15, 0.1]),
    'product_price': np.random.exponential(scale=100, size=n_records) + 20,
    'quantity': np.random.poisson(lam=2) + 1 for _ in range(n_records),
    'discount_pct': np.random.choice([0, 5, 10, 15, 20], n_records, p=[0.5, 0.2, 0.15, 0.1, 0.05]),
    'customer_age': np.random.normal(38, 12, n_records).astype(int),
    'is_returning_customer': np.random.choice([True, False], n_records, p=[0.3, 0.7]),
    'device_type': np.random.choice(['Mobile', 'Desktop', 'Tablet'], n_records, p=[0.6, 0.3, 0.1]),
    'region': np.random.choice(['North', 'South', 'East', 'West', 'Central'], n_records, p=[0.2, 0.25, 0.25, 0.2, 0.1])
})

# Fix age to be in reasonable range
df['customer_age'] = df['customer_age'].clip(18, 80)

# Calculate derived metrics
df['total_price'] = df['product_price'] * df['quantity']
df['final_price'] = df['total_price'] * (1 - df['discount_pct'] / 100)
df['date'] = df['timestamp'].dt.date

# Add some missing values realistically
missing_indices = np.random.choice(n_records, size=int(0.02 * n_records), replace=False)
df.loc[missing_indices[:50], 'customer_age'] = np.nan
df.loc[missing_indices[50:], 'discount_pct'] = np.nan

print("Dataset Overview:")
print(f"Shape: {df.shape}")
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nFirst 5 rows:")
print(df.head())

print(f"\nData Types:\n{df.dtypes}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## Part 2: Comprehensive Data Quality Assessment

In [ ]:
# Missing value analysis
print("Missing Value Analysis:")
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percent': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percent', ascending=False)
print(missing_df)

# Cardinality analysis
print("\nCardinality Analysis:")
cardinality = pd.DataFrame({
    'Column': df.columns,
    'Unique_Values': df.nunique(),
    'Unique_Percent': (df.nunique() / len(df) * 100).round(2)
})
print(cardinality.sort_values('Unique_Values', ascending=False))

# Data type summary
print("\nData Type Distribution:")
print(df.dtypes.value_counts())

# Duplicate analysis
print(f"\nDuplicate Rows: {df.duplicated().sum()}")
print(f"Duplicate Customer-Product combinations: {df.duplicated(subset=['customer_id', 'product_category', 'timestamp']).sum()}")

## Part 3: Statistical Profiling & Distribution Analysis

In [ ]:
# Detailed statistical profiling
print("Statistical Profiling - Continuous Variables:")
continuous_cols = ['product_price', 'quantity', 'customer_age', 'final_price']

stats_df = df[continuous_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
print(stats_df.round(2))

# Advanced statistical measures
print("\nAdvanced Statistics:")
for col in continuous_cols:
    data = df[col].dropna()
    skewness = stats.skew(data)
    kurtosis = stats.kurtosis(data)
    coeff_var = data.std() / data.mean() * 100
    
    print(f"\n{col}:")
    print(f"  Skewness: {skewness:.3f} (Right-skewed)" if skewness > 0.5 else f"  Skewness: {skewness:.3f} (Left-skewed)" if skewness < -0.5 else f"  Skewness: {skewness:.3f} (Symmetric)")
    print(f"  Kurtosis: {kurtosis:.3f}")
    print(f"  Coefficient of Variation: {coeff_var:.2f}%")
    
    # Normality test
    stat, p_value = stats.normaltest(data)
    print(f"  Normality Test p-value: {p_value:.6f} {'(Not Normal)' if p_value < 0.05 else '(Normal)'}")

## Part 4: Categorical Analysis & Distribution

In [ ]:
# Categorical value distributions
categorical_cols = ['product_category', 'device_type', 'region', 'is_returning_customer']

print("Categorical Distribution Analysis:\n")
for col in categorical_cols:
    print(f"{col}:")
    dist = df[col].value_counts(dropna=False).to_frame(name='Count')
    dist['Percent'] = (dist['Count'] / len(df) * 100).round(2)
    dist['Cumulative_Percent'] = dist['Percent'].cumsum().round(2)
    print(dist)
    print()

## Part 5: Correlation & Feature Relationships

In [ ]:
# Correlation matrix
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr(method='pearson')

print("Correlation Matrix (Pearson):")
print(corr_matrix.round(3))

# Find strong correlations
print("\nStrong Correlations (|r| > 0.3):")
strong_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.3:
            strong_corr.append({
                'Var1': corr_matrix.columns[i],
                'Var2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

if strong_corr:
    strong_corr_df = pd.DataFrame(strong_corr).sort_values('Correlation', key=abs, ascending=False)
    print(strong_corr_df.to_string(index=False))
else:
    print("No strong correlations found.")

# Spearman correlation for non-linear relationships
spearman_corr = numeric_df.corr(method='spearman')
print("\nSpearman Correlation (for non-linear relationships):")
print(spearman_corr.round(3))

## Part 6: Outlier Detection & Anomaly Analysis

In [ ]:
def detect_outliers_iqr(data, column, multiplier=1.5):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return (data[column] < lower) | (data[column] > upper)

def detect_outliers_zscore(data, column, threshold=3):
    z = np.abs(stats.zscore(data[column].dropna()))
    return data[column].apply(lambda x: abs(stats.zscore([x])[0]) > threshold if not pd.isna(x) else False)

# Outlier analysis
print("Outlier Detection Results:\n")
for col in continuous_cols:
    iqr_outliers = detect_outliers_iqr(df, col).sum()
    zscore_outliers = detect_outliers_zscore(df, col).sum()
    
    print(f"{col}:")
    print(f"  IQR Method: {iqr_outliers} outliers ({iqr_outliers/len(df)*100:.2f}%)")
    print(f"  Z-Score Method: {zscore_outliers} outliers ({zscore_outliers/len(df)*100:.2f}%)")
    print()

# Extreme values
print("\nExtreme Values:")
print(f"Max transaction value: ${df['final_price'].max():.2f}")
print(f"99th percentile: ${df['final_price'].quantile(0.99):.2f}")
print(f"Top 5 most expensive transactions:")
print(df.nlargest(5, 'final_price')[['transaction_id', 'product_category', 'quantity', 'final_price']])

## Part 7: Segment Analysis & Business Insights

# Revenue by product category
print("Revenue Analysis by Category:")
category_revenue = df.groupby('product_category').agg(
    total_revenue=('final_price', 'sum'),
    avg_transaction=('final_price', 'mean'),
    transaction_count=('transaction_id', 'count'),
    unique_customers=('customer_id', 'nunique')
).sort_values('total_revenue', ascending=False)
category_revenue['percent_revenue'] = (category_revenue['total_revenue'] / category_revenue['total_revenue'].sum() * 100).round(2)
print(category_revenue.round(2))

# Revenue by region
print("\nRevenue Analysis by Region:")
region_revenue = df.groupby('region').agg(
    total_revenue=('final_price', 'sum'),
    avg_transaction=('final_price', 'mean'),
    transaction_count=('transaction_id', 'count')
).sort_values('total_revenue', ascending=False)
print(region_revenue.round(2))

# Customer segmentation
print("\nCustomer Type Analysis:")
customer_type = df.groupby('is_returning_customer').agg(
    total_revenue=('final_price', 'sum'),
    avg_transaction=('final_price', 'mean'),
    transaction_count=('transaction_id', 'count'),
    unique_customers=('customer_id', 'nunique')
)
customer_type.index = ['New Customers', 'Returning Customers']
print(customer_type.round(2))

### 7.2 Customer Lifetime Value & Segmentation

### 7.3 Time Series & Temporal Patterns

### 7.4 Device & Channel Analysis

In [ ]:
# Device performance
print("Device Performance Metrics:")
device_analysis = df.groupby('device_type').agg({
    'transaction_id': 'count',
    'final_price': ['sum', 'mean', 'std'],
    'customer_id': 'nunique',
    'discount_pct': 'mean'
}).round(2)
print(device_analysis)

# Conversion efficiency by device
print("\nConversion Efficiency:")
device_stats = df.groupby('device_type').agg({
    'final_price': ['sum', 'count']
})
device_stats.columns = ['total_revenue', 'transactions']
device_stats['avg_order_value'] = device_stats['total_revenue'] / device_stats['transactions']
print(device_stats.round(2))